## Section 1 — Ganache Connection

In [ ]:
from web3 import Web3

ganache_url = "http://127.0.0.1:7545"  # Ganache RPC port
web3 = Web3(Web3.HTTPProvider(ganache_url))  # HTTPProvider connects over HTTP to local Ganache

if not web3.is_connected():  # is_connected() returns False if Ganache is unreachable
    raise ConnectionError("Cannot connect to Ganache. Ensure Ganache is running on port 7545.")

print("Connected to Ganache:", web3.is_connected())
print("Latest block number:", web3.eth.block_number)  # confirms live blockchain data is reachable

Connected to Ganache: True
Latest block number: 1


## Section 2 — Contract Load

In [2]:
import json

with open("../contracts/IoTDataStorage_compData.json") as f:  # load latest Remix compilation artifact
    artifact = json.load(f)

contract_config = {
    "address": "0x323b0a80f886F2dB2bfE0fb51Af157126098d8f5",  # update after each Remix redeployment
    "abi": artifact["abi"]  # ABI extracted from compiled artifact
}

iot_contract = web3.eth.contract(
    address=web3.to_checksum_address(contract_config["address"]),  # checksummed format required by Web3.py
    abi=contract_config["abi"]
)

web3.eth.default_account = web3.eth.accounts[0]  # accounts[0] is the deployer — matches the contract owner

print("Contract loaded at:", contract_config["address"])
print("Default account:", web3.eth.default_account)

pre_write_count = iot_contract.functions.iotRecordCount().call()  # .call() reads without a transaction — iotRecordCount() is the actual contract function
print("Records before write:", pre_write_count)

Contract loaded at: 0x323b0a80f886F2dB2bfE0fb51Af157126098d8f5
Default account: 0x2bb2598F2E92a38af5824d88D64Facf3FE0Eb8e8
Records before write: 0


## Section 3 — Dummy Test Transaction

In [3]:
# Step 1 — register a test shipment (required before any storeData() call)
reg_tx_hash = iot_contract.functions.registerShipment(
    "TEST-001",          # rfidTag — unique identifier for this test shipment
    0,                   # GoodsCategory enum index: 0 = DeepFreeze
    "Manila",            # origin
    "Cebu",              # destination
    1,                   # packageCount
    "VH-TEST",           # vehicleId
    "DR-TEST"            # driverId
).transact()  # .transact() writes to blockchain — returns tx_hash as HexBytes

web3.eth.wait_for_transaction_receipt(reg_tx_hash)  # block until shipment registration is mined
print("Test shipment registered. TX:", reg_tx_hash.hex())  # .hex() converts HexBytes to readable string

# Step 2 — store a dummy IoT record against the registered shipment
tx_hash = iot_contract.functions.storeData(
    "READ-TEST-001",     # readingId
    "TEST-001",          # rfidTag — must match the registered shipment above
    "DEV-TEST",          # deviceId
    "GPS",               # deviceType
    "latitude",          # dataType
    "14.5995"            # dataValue
).transact()  # .transact() writes to blockchain — returns tx_hash as HexBytes

receipt = web3.eth.wait_for_transaction_receipt(tx_hash)  # block until transaction is mined
print("Dummy record stored. TX:", tx_hash.hex())
print("Transaction status:", receipt.status)  # 1 = success, 0 = failed

# Step 3 — read back the dummy record to confirm it landed on-chain
dummy_record = iot_contract.functions.iotRecords(0).call()  # public array auto-getter — index 0 = first record
print("Record at index 0:", dummy_record)

Test shipment registered. TX: 9494a1450561c22c6b89a12b1df7438164d3e361b2037d160b4754d9108f9ada
Dummy record stored. TX: 6b855e303e9afa30b98b3cb5249e99c26e805ee7415e588e3c2048b2cf53213b
Transaction status: 1
Record at index 0: [1779435065, 'READ-TEST-001', 'TEST-001', 'DEV-TEST', 'GPS', 'latitude', '14.5995', '0x2bb2598F2E92a38af5824d88D64Facf3FE0Eb8e8']


## Section 3b — Type-Specific Record Helpers

In [4]:
def store_gps_record(rfid_tag, device_id, data_value):
    """Store a GPS location record on-chain via storeGPS()."""
    lat, lng = data_value.split(",")  # splits "14.60,120.98" into two variables in one line
    lat = lat.strip()                  # removes any accidental whitespace around the value
    lng = lng.strip()
    tx = iot_contract.functions.storeGPS(
        str(rfid_tag), str(device_id), lat, lng
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_temperature_record(rfid_tag, device_id, data_value):
    """Store a temperature reading on-chain via storeTemperature()."""
    temp_int = int(float(data_value) * 10)  # "12.2" → 12.2 → 122.0 → 122 — int16-safe encoding
    tx = iot_contract.functions.storeTemperature(
        str(rfid_tag), str(device_id), temp_int
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_rfid_record(rfid_tag, device_id, data_value):
    """Store an RFID scan status on-chain via storeRFIDScan()."""
    tx = iot_contract.functions.storeRFIDScan(
        str(rfid_tag), str(device_id), str(data_value)
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_generic_record(reading_id, rfid_tag, device_id, data_type, data_value):
    """Fallback: store any unrecognized data type via storeData()."""
    tx = iot_contract.functions.storeData(
        str(reading_id), str(rfid_tag), str(device_id),
        str(data_type), str(data_type), str(data_value)
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

## Section 4 — CSV Load + Bulk Write

In [5]:
import pandas as pd
import time

CATEGORY_MAP = {
    "Deep Freeze": 0, "Frozen": 1, "Chill/Refrigerated": 2,
    "Pharma": 3, "Cool-Chain": 4, "Dry Goods": 5,
    "Electronics": 6, "Clothing": 7, "Industrial": 8
}

def run_bulk_write():
    """Register all shipments from CSV, then write all IoT records with type routing."""
    # --- Shipment Registration ---
    ship_df = pd.read_csv("../data/shipment_registry.csv")
    print(f"Registering {len(ship_df)} shipments...")
    for _, ship in ship_df.iterrows():
        reg_hash = iot_contract.functions.registerShipment(
            ship["rfid_tag"],
            CATEGORY_MAP[ship["goods_category"]],
            ship["origin"], ship["destination"],
            int(ship["package_count"]),
            ship["vehicle_id"], ship["driver_id"]
        ).transact()
        web3.eth.wait_for_transaction_receipt(reg_hash)
        time.sleep(0.5)  # 0.5s — down from 1.0s, still prevents nonce conflicts
    print("All shipments registered.\n")

    # --- IoT Bulk Write ---
    iot_df = pd.read_csv("../data/iot_data.csv")
    print(f"Loaded {len(iot_df)} IoT records. Starting bulk write...")

    for index, row in iot_df.iterrows():
        try:
            data_type  = row["data_type"]
            rfid_tag   = row["rfid_tag"]
            device_id  = row["device_id"]
            data_value = str(row["data_value"])

            if data_type == "GPS":
                receipt = store_gps_record(rfid_tag, device_id, data_value)
            elif data_type == "Temperature":
                receipt = store_temperature_record(rfid_tag, device_id, data_value)
            elif data_type == "RFID":
                receipt = store_rfid_record(rfid_tag, device_id, data_value)
            else:
                receipt = store_generic_record(
                    row["reading_id"], rfid_tag, device_id, data_type, data_value
                )

            print(f"{data_type} | {rfid_tag} | {data_value} | Txn: {receipt.transactionHash.hex()}")
            # receipt.transactionHash is HexBytes — .hex() converts it to a readable string
            time.sleep(0.5)

        except Exception as e:  # catches any error for this row — loop continues instead of stopping
            print(f"Row {index} failed: {e}")

    print("\nBulk write complete.")

run_bulk_write()

Registering 30 shipments...
All shipments registered.

Loaded 329 IoT records. Starting bulk write...
Temperature | RFID-011 | 12.2 | Txn: 8ad35a9ffc6760034cbad1f973c9fcfc7bf2373d50596d0f2dda541c04959fba
GPS | RFID-020 | 14.601956,120.989036 | Txn: e6cc9b5b242a6b0e221538b0eb34534e7a28f77378d099f11d385ca62a0da358
GPS | RFID-011 | 14.620032,120.962165 | Txn: 05e156eabb7b112bf548e7f4c6af3a721d94c5a0c9649dbda693fe348ed94a82
RFID | RFID-011 | VERIFIED | Txn: 4a8d4bd9782a50b7cce38dbb4540bdbc39776521f4ceeeb91626628a3d93f105
GPS | RFID-020 | 14.595757,120.981906 | Txn: cdb433144af63853cbf2c9fe267050212335fd5193413dd5528e305ea2e19e73
GPS | RFID-028 | 14.374937,121.038127 | Txn: 3020cceb166f55ded8252fb88a5fd152686d536f3aa0bb23f7dea8ee4424089d
RFID | RFID-020 | VERIFIED | Txn: fe4e08a2b0f579604f47d9a3099e289eb4309fe3ee54e53ac6603c295ff269d3
Temperature | RFID-011 | 13.6 | Txn: 424ca273e0aae520c17105d96508b6c944ff0be17f18ae7da56357c85aa14bae
GPS | RFID-011 | 14.622406,120.970944 | Txn: c56057a96f3

## Section 5 — Verification

In [6]:
def run_verification():
    """Read all 5 on-chain counters and verify the first registered shipment."""
    print("=== On-Chain Record Counts ===")
    print(f"  Shipments registered : {iot_contract.functions.shipmentCount().call()}")
    print(f"  Generic IoT records  : {iot_contract.functions.iotRecordCount().call()}")
    print(f"  GPS records          : {iot_contract.functions.gpsRecordCount().call()}")
    print(f"  Temperature records  : {iot_contract.functions.tempRecordCount().call()}")
    print(f"  RFID scan records    : {iot_contract.functions.rfidRecordCount().call()}")

    first_tag = iot_contract.functions.getAllRFIDTags().call()[0]
    # getAllRFIDTags() returns a Python list of all rfidTag strings — [0] picks the first

    shipment = iot_contract.functions.getShipment(first_tag).call()
    # getShipment() returns a Shipment struct as a tuple — fields match struct declaration order

    print(f"\n=== First Shipment on Chain ({first_tag}) ===")
    print(f"  Origin      : {shipment[2]}")   # Shipment struct index 2 = origin
    print(f"  Destination : {shipment[3]}")   # Shipment struct index 3 = destination
    print(f"  Category    : {shipment[1]}")   # Shipment struct index 1 = category (enum int)
    print(f"  Vehicle ID  : {shipment[5]}")   # Shipment struct index 5 = vehicleId
    print(f"  Registered  : {shipment[7]}")   # Shipment struct index 7 = registeredAt (unix timestamp)

run_verification()

=== On-Chain Record Counts ===
  Shipments registered : 31
  Generic IoT records  : 1
  GPS records          : 150
  Temperature records  : 95
  RFID scan records    : 84

=== First Shipment on Chain (TEST-001) ===
  Origin      : Manila
  Destination : Cebu
  Category    : 0
  Vehicle ID  : VH-TEST
  Registered  : 1779435065
